In [45]:
import pandas as pd
import numpy as np

import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [46]:
data = pd.read_csv(r'WA_Fn-UseC_-Telco-Customer-Churn.csv')

data.shape

(7043, 21)

In [47]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [48]:
data.duplicated()

0       False
1       False
2       False
3       False
4       False
        ...  
7038    False
7039    False
7040    False
7041    False
7042    False
Length: 7043, dtype: bool

In [49]:
data.isna().sum().sum()

0

In [50]:
data.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [51]:
# Optimize the memory usage

start_memory = data.memory_usage(deep=True).sum()/1024**2
print(f"Memory Before Optimization : {start_memory:.2f} Mb")

for col in data.columns:
    coltype = data[col].dtype

    if pd.api.types.is_integer_dtype(coltype):
        cmin = data[col].min()
        cmax = data[col].max()

        if cmin >= np.iinfo(np.int8).min and cmax <= np.iinfo(np.int8).max:
            data[col] = data[col].astype(np.int8)
        elif cmin >= np.iinfo(np.int16).min and cmax <= np.iinfo(np.int16).max:
            data[col] = data[col].astype(np.int16)
        elif cmin >= np.iinfo(np.int32).min and cmax <= np.iinfo(np.int32).max:
            data[col] = data[col].astype(np.int32)
        else:
            data[col] = data[col].astype(np.int64)
            
    if pd.api.types.is_float_dtype(coltype):
        cmin = data[col].min()
        cmax = data[col].max()  

        if cmin >= np.finfo(np.float16).min and cmax <= np.finfo(np.float16).max:
            data[col] = data[col].astype(np.float16)
        elif cmin >= np.finfo(np.float32).min and cmax <= np.finfo(np.float32).max:
            data[col] = data[col].astype(np.float32)
        else: 
            data[col] = data[col].astype(np.float64)

end_memory = data.memory_usage(deep=True).sum()/1024**2

per = (end_memory/start_memory)*100

print(f"Memory after Optimization : {end_memory:.2f} Mb \nMemory Reduced By : {per:.2f}%")

Memory Before Optimization : 6.82 Mb
Memory after Optimization : 6.69 Mb 
Memory Reduced By : 98.03%


In [56]:
# Label Encoding

le = LabelEncoder()

for i in data.select_dtypes(include='object').columns:
    data[i] = data[i].astype(str)

    le.fit(data[i])
    transformed = le.transform(data[i])
    data[i] = transformed

In [58]:
data.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,5375,0,0,1,0,1,0,1,0,0,...,0,0,0,0,0,1,2,29.84375,2505,0
1,3962,1,0,0,0,34,1,0,0,2,...,2,0,0,0,1,0,3,56.93750,1466,0
2,2564,1,0,0,0,2,1,0,0,2,...,0,0,0,0,0,1,3,53.84375,157,1
3,5535,1,0,0,0,45,0,1,0,2,...,2,2,0,0,1,0,0,42.31250,1400,0
4,6511,0,0,0,0,2,1,0,1,0,...,0,0,0,0,0,1,2,70.68750,925,1


In [60]:
data['Churn'].value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

In [62]:
# Split the data to train and test

X = data.drop('Churn',axis=1)
y = data['Churn']

In [64]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2)

In [68]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier()
rf = RandomForestClassifier()

dt.fit(X_train,y_train)
y_pred1 = dt.predict(X_test)

rf.fit(X_train,y_train)
y_pred2 = rf.predict(X_test)

In [74]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report

print(f"Accuracy of RandomForestClassifier : {(accuracy_score(y_test,y_pred2)*100):.2f} %")
print("===============================================")
print(f"Accuracy of DecisionTreeClassifier : {(accuracy_score(y_test,y_pred1)*100):.2f} %")

Accuracy of RandomForestClassifier : 78.57 %
Accuracy of DecisionTreeClassifier : 69.06 %


In [78]:
print(f"Classification Report of RandomForestClassifier : \n{classification_report(y_test,y_pred2)}")
print("===============================================")
print(f"Classification Report of DecisionTreeClassifier : \n{classification_report(y_test,y_pred1)} ")

Classification Report of RandomForestClassifier : 
              precision    recall  f1-score   support

           0       0.82      0.91      0.86      1030
           1       0.65      0.45      0.53       379

    accuracy                           0.79      1409
   macro avg       0.73      0.68      0.70      1409
weighted avg       0.77      0.79      0.77      1409

Classification Report of DecisionTreeClassifier : 
              precision    recall  f1-score   support

           0       0.80      0.77      0.79      1030
           1       0.43      0.46      0.45       379

    accuracy                           0.69      1409
   macro avg       0.61      0.62      0.62      1409
weighted avg       0.70      0.69      0.69      1409
 


In [80]:
print(f"Confusion Matrix of RandomForestClassifier : \n{confusion_matrix(y_test,y_pred2)}")
print("===============================================")
print(f"Confusion Matrix of DecisionTreeClassifier : \n{confusion_matrix(y_test,y_pred1)} ")

Confusion Matrix of RandomForestClassifier : 
[[937  93]
 [209 170]]
Confusion Matrix of DecisionTreeClassifier : 
[[798 232]
 [204 175]] 
